In [72]:
# numrerical & analytical libraries
import numpy as np 
import pandas as pd

# visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# modeling libraries
from statsmodels.stats.weightstats import ttest_ind
from scipy.stats import t

In [73]:
# PARAMETERS
control_n = 1000
test_n = 1000
test_segment = 100  # For grouping into 10 tests

# Generate random conversion rates for each test group
control_conversion_rates = np.random.uniform(0.18, 0.25, size=control_n)
test_conversion_rates = np.random.uniform(0.18, 0.40, size=test_n)

# Generate control data
control_bet_price = np.random.normal(loc=75, scale=10, size=control_n).round(2)
control_bet_placed = np.random.binomial(1, control_conversion_rates)

c_df = pd.DataFrame({
    'user_id': [f'C{i}' for i in range(control_n)],
    'group': 'control_group',
    'bet_priced': control_bet_price,
    'bet_placed': control_bet_placed
})

# Generate test data
test_bet_price = np.random.normal(loc=100, scale=12, size=test_n).round(2)
test_bet_placed = np.random.binomial(1, test_conversion_rates)

t_df = pd.DataFrame({
    'user_id': [f'T{i}' for i in range(test_n)],
    'group': 'test_group',
    'bet_priced': test_bet_price,
    'bet_placed': test_bet_placed
})

# Add test number labels (e.g., Test 1, Test 2, ..., Test 10)
c_df['test_number'] = (c_df.index // test_segment) + 1
t_df['test_number'] = (t_df.index // test_segment) + 1

t_df

,user_id,group,bet_priced,bet_placed,test_number
0,T0,test_group,115.91,0,1
1,T1,test_group,96.26,0,1
2,T2,test_group,109.83,0,1
3,T3,test_group,77.77,0,1
4,T4,test_group,78.35,1,1
...,...,...,...,...,...
995,T995,test_group,79.82,0,10
996,T996,test_group,108.40,0,10
997,T997,test_group,102.54,0,10
998,T998,test_group,109.56,1,10


In [74]:
c_df

,user_id,group,bet_priced,bet_placed,test_number
0,C0,control_group,75.86,0,1
1,C1,control_group,76.13,1,1
2,C2,control_group,60.01,0,1
3,C3,control_group,73.01,0,1
4,C4,control_group,69.87,1,1
...,...,...,...,...,...
995,C995,control_group,87.08,1,10
996,C996,control_group,87.18,0,10
997,C997,control_group,68.62,0,10
998,C998,control_group,76.66,0,10


| **Cohen’s d** | **Interpretation**                          | **When to Act / What It Means**                                         |
| ------------- | ------------------------------------------- | ----------------------------------------------------------------------- |
| **≥ 6.0**     | 🚨 **Severe deviation (Critical Breach)**   | Immediate attention — likely failure in pricing logic or margin config. |
| **≥ 4.0**     | ⚠️ **High deviation (Margin Breach Risk)**  | Check if market margin config is missing, disabled, or overridden.      |
| **≥ 2.5**     | ✅ **Controlled volatility (Expected High)** | Volatility is high, but within anticipated bounds for active markets.   |
| **≥ 1.5**     | 🟢 **Normal variation (Integrated market)** | Standard differences between test and control — no action needed.       |
| **< 1.5**     | ⚪ **Noise or trivial difference**           | Not practically meaningful — ignore in ops reporting.                   |


In [75]:
def interpret_cohens_d(d_value):
    """
    Tailored Cohen's d interpretation for sports betting market pricing.
    Flags only meaningful deviations based on operational pricing logic.
    """
    abs_d = abs(d_value)

    if abs_d >= 6.0:
        return "Severe deviation (Critical Breach)"
    elif abs_d >= 4.0:
        return "High deviation (Margin Breach Risk)"
    elif abs_d >= 2.5:
        return "Controlled volatility (Expected High)"
    elif abs_d >= 1.5:
        return "Normal variation (Integrated market)"
    else:
        return "Noise or trivial difference"


In [76]:
def one_tailed_t_test_independent(sample1, sample2, alpha=0.05):
    """
    Performs a one-tailed INDEPENDENT samples t-test, calculates Cohen's d,
    raw/percentage differences, and provides business threshold interpretation.
    Ha: mean(sample2 - test group) > mean(sample1 - control group)

    Args:
        sample1 (array-like): First independent sample (control group)
        sample2 (array-like): Second independent sample (test group)
        alpha (float, optional): Significance level. Defaults to 0.05.

    Returns:
        tuple: (t_statistic, p_value, degrees_of_freedom, reject_null, critical_value,
                cohens_d, cohens_d_interpretation, raw_price_difference, percent_difference_str)
                - t_statistic: The calculated t-statistic.
                - p_value: The one-tailed p-value.
                - degrees_of_freedom: The degrees of freedom for the t-distribution.
                - reject_null: True if the null hypothesis is rejected, False otherwise.
                - critical_value: The critical t value at the given alpha level.
                - cohens_d: The calculated Cohen's d effect size.
                - cohens_d_interpretation: String interpretation based on business thresholds.
                - raw_price_difference: The absolute difference in means (mean2 - mean1).
                - percent_difference_str: String representation of percentage difference,
                                          handles division by zero.
    """
    # Convert samples to numpy arrays for easier calculations
    sample1 = np.array(sample1)
    sample2 = np.array(sample2)

    # Calculate basic statistics
    mean1, mean2 = np.mean(sample1), np.mean(sample2)
    std1, std2 = np.std(sample1, ddof=1), np.std(sample2, ddof=1) # ddof=1 for sample standard deviation
    n1, n2 = len(sample1), len(sample2)

    # --- Calculate Raw and Percentage Differences ---
    raw_price_difference = mean2 - mean1

    # Calculate percentage difference, handling division by zero for mean1
    if mean1 != 0:
        percent_difference = (raw_price_difference / mean1) * 100
        percent_difference_str = f"{percent_difference:.2f}%"
    else:
        percent_difference_str = "Undefined (Control mean is zero)" # Or handle as per business logic

    # Calculate pooled standard deviation for Cohen's d
    pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))

    # Calculate Cohen's d
    cohens_d = raw_price_difference / pooled_std # Using raw_price_difference directly

    # Get the interpretation of Cohen's d
    cohens_d_interpretation = interpret_cohens_d(cohens_d)

    # Determine alternative for statsmodels: Ha: mean(sample2) > mean(sample1) implies mean(sample1) - mean(sample2) < 0
    sm_alt = 'smaller'

    # Perform the t-test using statsmodels
    t_statistic, p_value, degrees_of_freedom = ttest_ind(sample1, sample2, alternative=sm_alt, usevar='pooled')

    # Critical value for a left-tailed test
    critical_value = t.ppf(alpha, df=degrees_of_freedom)

    # Decision Rule
    reject_null = (p_value < alpha) and (t_statistic < 0)

    # Print results to console
    print("\n--- T-Test Results ---")
    if reject_null:
        print(f"Null hypothesis rejected: sample2's mean is significantly greater than sample1's mean (t = {t_statistic:.4f} < critical value = {critical_value:.4f}, p = {p_value:.4f} < alpha = {alpha})")
    else:
        print(f"Failed to reject null hypothesis: not enough evidence that sample2's mean is greater than sample1's mean (t = {t_statistic:.4f}, critical value = {critical_value:.4f}, p = {p_value:.4f})")

    print(f"Raw Difference (Test - Control): {raw_price_difference:.2f}")
    print(f"Percentage Difference (relative to Control): {percent_difference_str}")
    print(f"Cohen's d (Effect Size): {cohens_d:.4f} ({cohens_d_interpretation})")
    print("---------------------\n")

    # Update global results DataFrame
    global results # callable outside function call (global scope)
    results = pd.DataFrame({
        'alpha': alpha,
        'test_statistic': t_statistic,
        'degrees_of_freedom': degrees_of_freedom,
        'p_value': p_value,
        'critical_value': critical_value,
        'decision_rule_p_value': reject_null,
        'cohens_d': cohens_d,
        'cohens_d_interpretation': cohens_d_interpretation,
        'raw_price_difference': raw_price_difference,
        'percentage_difference': percent_difference_str # Store as string for easy reporting
    }, index=[0])

    return t_statistic, p_value, degrees_of_freedom, reject_null, critical_value, cohens_d, cohens_d_interpretation, raw_price_difference, percent_difference_str


In [77]:
def run_all_segmented_tests(c_df, t_df, alpha=0.05):
    """
    Loops through each unique test_number (1–10) and runs one_tailed_t_test_independent().

    Args:
        c_df (pd.DataFrame): Control group with 'bet_priced' and 'test_number'
        t_df (pd.DataFrame): Test group with 'bet_priced' and 'test_number'
        alpha (float): Significance level (default 0.05)

    Returns:
        pd.DataFrame: Combined results across all segments
    """
    all_results = []

    for test_num in sorted(c_df['test_number'].unique()):
        # Extract 100 rows from control and test for this segment
        control_sample = c_df[c_df['test_number'] == test_num]['bet_priced']
        test_sample = t_df[t_df['test_number'] == test_num]['bet_priced']

        # Run the test using your original function
        t_stat, p_val, dfree, reject, crit_val, d, d_interp, raw_diff, pct_diff = one_tailed_t_test_independent(
            control_sample, test_sample, alpha=alpha
        )

        all_results.append({
            'test_number': test_num,
            'alpha': alpha,
            'test_statistic': t_stat,
            'degrees_of_freedom': dfree,
            'p_value': p_val,
            'critical_value': crit_val,
            'reject_null': reject,
            'cohens_d': d,
            'cohens_d_interpretation': d_interp,
            'raw_price_difference': raw_diff,
            'percentage_difference': pct_diff
        })

    return pd.DataFrame(all_results)


In [78]:
results_df = run_all_segmented_tests(c_df, t_df, alpha=0.05)
results_df


--- T-Test Results ---
Null hypothesis rejected: sample2's mean is significantly greater than sample1's mean (t = -14.6928 < critical value = -1.6526, p = 0.0000 < alpha = 0.05)
Raw Difference (Test - Control): 24.22
Percentage Difference (relative to Control): 32.11%
Cohen's d (Effect Size): 2.0779 (Normal variation (Integrated market))
---------------------


--- T-Test Results ---
Null hypothesis rejected: sample2's mean is significantly greater than sample1's mean (t = -17.0138 < critical value = -1.6526, p = 0.0000 < alpha = 0.05)
Raw Difference (Test - Control): 24.57
Percentage Difference (relative to Control): 32.15%
Cohen's d (Effect Size): 2.4061 (Normal variation (Integrated market))
---------------------


--- T-Test Results ---
Null hypothesis rejected: sample2's mean is significantly greater than sample1's mean (t = -16.6434 < critical value = -1.6526, p = 0.0000 < alpha = 0.05)
Raw Difference (Test - Control): 25.65
Percentage Difference (relative to Control): 33.97%
Co

,test_number,alpha,test_statistic,degrees_of_freedom,p_value,critical_value,reject_null,cohens_d,cohens_d_interpretation,raw_price_difference,percentage_difference
0,1,0.05,-14.692810,198.0,7.777044e-34,-1.652586,True,2.077877,Normal variation (Integrated market),24.2160,32.11%
1,2,0.05,-17.013756,198.0,6.713161e-41,-1.652586,True,2.406108,Normal variation (Integrated market),24.5666,32.15%
2,3,0.05,-16.643398,198.0,8.801137e-40,-1.652586,True,2.353732,Normal variation (Integrated market),25.6524,33.97%
3,4,0.05,-16.561288,198.0,1.559254e-39,-1.652586,True,2.342120,Normal variation (Integrated market),25.2039,33.51%
4,5,0.05,-17.268566,198.0,1.150043e-41,-1.652586,True,2.442144,Normal variation (Integrated market),25.5247,33.65%
5,6,0.05,-18.237304,198.0,1.479537e-44,-1.652586,True,2.579144,Controlled volatility (Expected High),26.9395,36.57%
6,7,0.05,-15.513015,198.0,2.401201e-36,-1.652586,True,2.193872,Normal variation (Integrated market),25.9837,35.15%
7,8,0.05,-13.832846,198.0,3.392398e-31,-1.652586,True,1.956260,Normal variation (Integrated market),22.2861,29.61%
8,9,0.05,-17.622868,198.0,9.983107e-43,-1.652586,True,2.492250,Normal variation (Integrated market),25.8774,34.92%
9,10,0.05,-15.554057,198.0,1.799388e-36,-1.652586,True,2.199676,Normal variation (Integrated market),23.7267,31.60%
